# 06 - Run Classical Baseline Optimizers (CMA-ES, DE, PSO)

This notebook:
1. Defines classical baseline algorithms: **CMA-ES**, **Differential Evolution (DE)**, and **Particle Swarm Optimization (PSO)**.
2. Runs each baseline **N=10 independent times** on target BBOB problems (`f1, f8, f11, f15, f21`) at `dim=5`.
3. Uses the **exact same noisy `BBOBProblem` wrapper** (`MultiplicativeNoiseStrategy(0.05)`) as LLaMEA experiments.
4. Attaches IOH Analyzer via `problem.attach_analyzer(...)` to output IOH `.dat` performance files to `data/ioh_logs/f{p_id}_5D_std0.05/{cmaes|de|pso}/`.

In [ ]:
import sys
import numpy as np
import cma
from scipy.optimize import differential_evolution
from pathlib import Path

# Add src to path
PROJECT_ROOT = Path('../').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from infra.problems.bbob import BBOBProblem
from domain.services.noise_strategy import MultiplicativeNoiseStrategy

TARGET_PROBLEMS = [1, 8, 11, 15, 21]
IOH_LOGS_DIR = PROJECT_ROOT / 'data' / 'ioh_logs'
N_RUNS = 10
BUDGET = 100000
NOISE_STD = 0.05
DIM = 5

print(f'Target Problems: {TARGET_PROBLEMS}')
print(f'Runs per problem: {N_RUNS}')
print(f'Budget: {BUDGET} evaluations')

## 1. Algorithm Implementations

In [ ]:
def run_cmaes(problem, dim: int, budget: int):
    """Run CMA-ES algorithm on problem with specified evaluation budget."""
    x0 = [0.0] * dim
    sigma0 = 2.0
    opts = {'bounds': [-5.0, 5.0], 'verbose': -9, 'maxfevals': budget}
    es = cma.CMAEvolutionStrategy(x0, sigma0, opts)
    while not es.stop():
        solutions = es.ask()
        es.tell(solutions, [problem(x) for x in solutions])
        if problem.evaluations >= budget:
            break
    return es.result.xbest, es.result.fbest

def run_de(problem, dim: int, budget: int):
    """Run Differential Evolution (DE) on problem with specified evaluation budget."""
    bounds = [(-5.0, 5.0)] * dim
    maxiter = max(1, budget // (15 * dim))
    
    def obj_fn(x):
        if problem.evaluations >= budget:
            raise StopIteration('Budget exhausted')
        return problem(x)
        
    try:
        res = differential_evolution(obj_fn, bounds, maxiter=maxiter, seed=None)
        return res.x, res.fun
    except StopIteration:
        return problem.optimum_x, problem.true_optimum

def run_pso(problem, dim: int, budget: int, n_particles: int = 30, w: float = 0.729, c1: float = 1.49445, c2: float = 1.49445):
    """Run Particle Swarm Optimization (PSO) on problem with specified evaluation budget."""
    lb, ub = problem.lb, problem.ub
    X = np.random.uniform(lb, ub, (n_particles, dim))
    V = np.random.uniform(-abs(ub - lb), abs(ub - lb), (n_particles, dim)) * 0.1
    
    pbest_X = X.copy()
    pbest_y = np.array([problem(x) for x in X])
    
    gbest_idx = np.argmin(pbest_y)
    gbest_X = pbest_X[gbest_idx].copy()
    gbest_y = pbest_y[gbest_idx]
    
    evals = n_particles
    while evals < budget:
        r1 = np.random.rand(n_particles, dim)
        r2 = np.random.rand(n_particles, dim)
        V = w * V + c1 * r1 * (pbest_X - X) + c2 * r2 * (gbest_X - X)
        X = np.clip(X + V, lb, ub)
        
        for i in range(n_particles):
            if evals >= budget:
                break
            y = problem(X[i])
            evals += 1
            if y < pbest_y[i]:
                pbest_y[i] = y
                pbest_X[i] = X[i].copy()
                if y < gbest_y:
                    gbest_y = y
                    gbest_X = X[i].copy()
                    
    return gbest_X, gbest_y

## 2. Benchmark Execution Loop

In [ ]:
BASELINES = {
    'cmaes': run_cmaes,
    'de': run_de,
    'pso': run_pso,
}

for p_id in TARGET_PROBLEMS:
    out_dir = IOH_LOGS_DIR / f'f{p_id}_{DIM}D_std{NOISE_STD}'
    out_dir.mkdir(parents=True, exist_ok=True)
    
    for algo_name, runner_fn in BASELINES.items():
        print(f'\n=== Running {algo_name.upper()} on f{p_id} (N={N_RUNS} runs, Budget={BUDGET}) ===')
        
        problem = BBOBProblem(
            problem_id=p_id,
            dim=DIM,
            instance_id=1,
            noise_strategy=MultiplicativeNoiseStrategy(NOISE_STD),
        )
        
        problem.attach_analyzer(
            log_dir=out_dir,
            folder_name=algo_name,
            algorithm_name=algo_name.upper()
        )
        
        for run_idx in range(1, N_RUNS + 1):
            problem.reset()
            try:
                best_x, best_y = runner_fn(problem, DIM, BUDGET)
            except Exception as e:
                print(f'  Run {run_idx:2d}/{N_RUNS} error: {e}')
                continue
                
            clean_val = problem.eval_clean(problem.clip(best_x))
            clean_err = abs(clean_val - problem.true_optimum)
            print(f'  Run {run_idx:2d}/{N_RUNS}: final clean error = {clean_err:.6e}')
            
        # Safely close logger
        problem.close_logger()
        print(f'  Saved IOH logs for {algo_name.upper()} to {out_dir / algo_name}')

print('\nAll baseline runs complete!')